# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali-Haider987/alihaider-flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Contract — lane: Content Decline / Refresh Prioritization

*(Skills loaded for this notebook: `writing-data-contracts`, `querying-big-datasets`, and `flyrank/flyrank-data` per `skills/README.md`.)*

**One row (unit of analysis) = one content item, for one client, summarized over one month.**
The warehouse's native grain — per the dataset card — is **"one row per report date, pseudonymized client, and pseudonymized content item"** (`fact_content_daily_performance`, 78,835,655 rows total across the full release). I aggregate that daily grain up to a monthly content-level row here, because the decision I'm supporting — *"should this page be queued for refresh?"* — is made per page, not per page-per-day.

**Tables used** (grain as documented on the dataset card):
- `fact_content_daily_performance` — one row per (report date, client, content item). Primary source for features and the label. Partitioned by `month=YYYY-MM`.
- `fact_content_query_90d` — one row per (client, content item, **query hash**) over a fixed trailing 90-day window. I use it for content-level query-mix signals only (via `ANY_VALUE`, since those aggregates are repeated across every query row for a given content item).
- `dim_content` — one row per content item. Context + one static feature (word count).
- `dim_clients` — one row per client. Used only to check GA4-export availability, never fed to the model.

**Time window:** a single mid-panel month, **`2026-03`** (not the sealed final month `2026-06`, which the warehouse's own README flags as the natural outcome window of any past→future label and reserves as held-out). Within that month I split on the 15th:
- **Decision moment:** end of day, 2026-03-15.
- **Feature window:** 2026-03-01 → 2026-03-15 ("first half") — fully observed *before* the decision moment.
- **Outcome window:** 2026-03-16 → 2026-03-31 ("second half") — the future relative to the decision moment; this is where the label comes from.

**Label / proxy:** `is_declining` = 1 if second-half GSC impressions fall below 80% of first-half GSC impressions. It's an *observed outcome* built from raw counts I compute myself — not one of FlyRank's own precomputed `trend_*` / `health_score` fields (those live in the separate `internship-lanes` dataset, and FlyRank's own card for that dataset explicitly flags them as **leakage-risk context, not default features**). Building my own label from raw impressions avoids inheriting that risk.

**One thing I deliberately exclude:** raw GA4 event-level rows and individual (non-aggregated) query strings. I only use pre-aggregated daily/90-day summaries. Event- and query-level detail is finer-grained than my unit of analysis needs, and query strings can carry near-identifying long-tail phrasing I don't need to touch for this lane.

In [4]:
# Setup: connect DuckDB to the hosted release (same pattern as notebook 03).
%pip -q install duckdb huggingface_hub

import os, getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Verification: confirm every table exists and see rough scale before claiming anything about it.
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

# Discover real column names rather than assuming them — this is the "verify, don't guess" habit
# the whole contract depends on.
def columns_of(table_sql):
    return [r[0] for r in con.sql(f'DESCRIBE SELECT * FROM {table_sql} LIMIT 0').fetchall()]

def resolve_col(table_sql, candidates):
    cols = columns_of(table_sql)
    for c in candidates:
        if c in cols:
            return c
    raise ValueError(f'None of {candidates} found. Actual columns: {cols}')

print('\nfact_daily columns:   ', columns_of(TABLES['fact_daily']))
print('dim_content columns:  ', columns_of(TABLES['dim_content']))
print('dim_clients columns:  ', columns_of(TABLES['dim_clients']))
print('fact_query_90d columns:', columns_of(TABLES['fact_query_90d']))


Paste your Hugging Face READ token (hf_...): ··········
dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows

fact_daily columns:    ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
dim_content columns:   ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'categor

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Field Buckets

### Feature
- `gsc_impressions` (first half: 2026-03-01 to 2026-03-15)
- `gsc_clicks` (first half: 2026-03-01 to 2026-03-15)
- `gsc_avg_position` (first half: 2026-03-01 to 2026-03-15)
- `content_visible_query_count` from `fact_content_query_90d`
- `rare_impressions_share` from `fact_content_query_90d`
- `anonymized_impressions_share` from `fact_content_query_90d`
- `word_count` from `dim_content`

**Why:**  
All features are fully observed at or before the decision date (2026-03-15).

---

### Label
- `is_declining` (derived from second-half impressions compared with first-half impressions)

**Why:**  
This is the target variable the model predicts and is never used as an input feature.

---

### Context
- `client_hash_id`
- `content_hash_id`
- `dim_clients.ga4_data_start`
- `dim_clients.gsc_data_start`
- `access_profile`

**Why:**  
These fields are used for joins, filtering, and data availability checks, but are not provided to the model as predictive signals.

---

### Excluded
- Raw GA4 event rows
- Individual query strings
- Sealed `fact_daily_sample` table (2026-06)
- FlyRank leakage-risk fields:
  - `trend_*`
  - `health_score`
  - `recommended_action`
  - `action_type`
- Any second-half column other than the one required to construct the label

**Why:**  
- Raw GA4 events and query strings are more granular than the project's unit of analysis.
- The 2026-06 sample is intentionally reserved as a future test set.
- FlyRank explicitly identifies `trend_*`, `health_score`, `recommended_action`, and `action_type` as leakage-risk variables.
- Any information from the outcome window (second half of the month) is excluded from predictors to prevent data leakage.

In [5]:
# Show the bucketing programmatically so it isn't just prose.
field_contract = pd.DataFrame([
    ('gsc_impressions (first half)',        'feature', 'observed by decision moment'),
    ('gsc_clicks (first half)',             'feature', 'observed by decision moment'),
    ('gsc_avg_position (first half)',       'feature', 'observed by decision moment'),
    ('content_visible_query_count',         'feature', 'trailing 90d snapshot, pre-decision'),
    ('rare_impressions_share',              'feature', 'trailing 90d snapshot, pre-decision'),
    ('anonymized_impressions_share',        'feature', 'trailing 90d snapshot, pre-decision'),
    ('word_count',                          'feature', 'static content attribute'),
    ('is_declining',                        'label',   'derived from second-half outcome'),
    ('client_hash_id / content_hash_id',    'context', 'join keys only'),
    ('gsc_data_start / ga4_data_start',     'context', 'availability filter only'),
    ('raw GA4 events',                      'excluded','finer grain than unit of analysis'),
    ('individual query strings',            'excluded','finer grain + long-tail sensitivity'),
    ('fact_daily_sample (2026-06, sealed)', 'excluded','held-out future month, not for label design'),
], columns=['field', 'bucket', 'reason'])
field_contract



,field,bucket,reason
0,gsc_impressions (first half),feature,observed by decision moment
1,gsc_clicks (first half),feature,observed by decision moment
2,gsc_avg_position (first half),feature,observed by decision moment
3,content_visible_query_count,feature,"trailing 90d snapshot, pre-decision"
4,rare_impressions_share,feature,"trailing 90d snapshot, pre-decision"
5,anonymized_impressions_share,feature,"trailing 90d snapshot, pre-decision"
6,word_count,feature,static content attribute
7,is_declining,label,derived from second-half outcome
8,client_hash_id / content_hash_id,context,join keys only
9,gsc_data_start / ga4_data_start,context,availability filter only


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Fact 1 the grain
*Claim (matching the dataset card's own description): `fact_content_daily_performance` is one row per (client, content item, day)  no duplicates.*

In [6]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()

print(f'Duplicate (client, content, day) combinations in month=2026-03: {len(grain_check)}')
print('Grain confirmed' if len(grain_check) == 0 else 'Grain claim is WRONG — investigate before trusting anything downstream.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (client, content, day) combinations in month=2026-03: 0
Grain confirmed


### Fact 2 — row count and date span for the slice
*Claim: `month=2026-03` is a clean, single-month slice with no stray dates leaking in from neighboring months. As a sanity check, the full release's `fact_content_daily_performance` totals 78,835,655 rows across all ~17 months for ~70 clients — so one month's slice here should land in a plausible fraction of that, not near zero and not near the full total.*

In [7]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

FULL_RELEASE_ROWS = 78_835_655  # from the dataset card, for sanity-checking this slice's scale
print(f"This month's rows are {span['n_rows'][0] / FULL_RELEASE_ROWS:.2%} of the full release's daily-fact row count.")
span

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

This month's rows are 12.48% of the full release's daily-fact row count.


,n_rows,min_date,max_date,n_clients,n_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


### Fact 3  availability, filtered with `IS TRUE`
*Not every client has a GA4 export configured. I build an explicit boolean from `dim_clients` and filter on it, rather than assuming the field already exists as a flag.*

In [8]:
avail = con.sql(f"""
    WITH flagged AS (
        SELECT client_hash_id,
               (ga4_data_start IS NOT NULL) AS has_ga4_export
        FROM {TABLES['dim_clients']}
    )
    SELECT
        COUNT(*) AS total_clients,
        COUNT(*) FILTER (WHERE has_ga4_export IS TRUE) AS clients_with_ga4
    FROM flagged
""").df()

avail['pct_with_ga4'] = (avail['clients_with_ga4'] / avail['total_clients']).round(3)
avail


,total_clients,clients_with_ga4,pct_with_ga4
0,104,51,0.49


### Five features (max), each tagged "knowable at decision moment because…"

1. **`imp_first_half`** — sum of `gsc_impressions`, 2026-03-01→15. *Knowable because it is fully observed history strictly before the 15th.*
2. **`clk_first_half`** — sum of `gsc_clicks`, 2026-03-01→15. *Same reason — closed historical window.*
3. **`pos_first_half`** — average `gsc_avg_position`, 2026-03-01→15. *Same reason — closed historical window.*
4. **`visible_queries`** (`content_visible_query_count` from `fact_content_query_90d`) — *Knowable because it's a trailing 90-day snapshot computed independently of the March outcome split; it reflects query mix built up before the decision moment, not the outcome itself.*
5. **`word_count`** (`dim_content`) — *Knowable because it's a static content attribute set when the page was published/last edited, with no dependency on any performance window at all.*

In [ ]:
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks      ELSE 0 END) AS clk_first_half,
           AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END)        AS pos_first_half,
           SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY 1, 2
    HAVING imp_first_half >= 20   -- floor: half-month volume is noisy below this
""").df()

q_col = resolve_col(TABLES['fact_query_90d'], ['content_visible_query_count'])
qsig = con.sql(f"""
    SELECT content_hash_id, ANY_VALUE({q_col}) AS visible_queries
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

wc_col = resolve_col(TABLES['dim_content'], ['word_count', 'content_word_count', 'wordcount'])
content_key = resolve_col(TABLES['dim_content'], ['content_hash_id'])
wc = con.sql(f"""
    SELECT {content_key} AS content_hash_id, {wc_col} AS word_count
    FROM {TABLES['dim_content']}
""").df()

data = feat.merge(qsig, on='content_hash_id', how='left').merge(wc, on='content_hash_id', how='left')
print(f'{len(data):,} content items with enough first-half history')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### The trap: add one label-derived column, watch the score jump, then remove it

The label is *defined by* the ratio of second-half to first-half impressions. If I feed that same ratio in as a "feature," the model isn't learning anything — it's just reading the answer key. This is the exact leakage lesson from notebook 02 (`trend_pct`), repeated here on real warehouse data.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

data['is_declining'] = (data['imp_second_half'] < 0.8 * data['imp_first_half']).astype(int)

honest_features = ['imp_first_half', 'clk_first_half', 'pos_first_half', 'visible_queries', 'word_count']
model_data = data.dropna(subset=honest_features).copy()
X, y = model_data[honest_features], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
honest_model = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f'Honest AUC (5 pre-decision features only): {honest_auc:.3f}')

# --- Now deliberately leak: feed in the exact ratio the label thresholds on ---
model_data['second_to_first_ratio'] = model_data['imp_second_half'] / model_data['imp_first_half']
leaky_features = honest_features + ['second_to_first_ratio']
Xl_tr, Xl_te, _, _ = train_test_split(model_data[leaky_features], y, test_size=0.25, random_state=42, stratify=y)
leaky_model = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42).fit(Xl_tr, y_tr)
leaky_auc = roc_auc_score(y_te, leaky_model.predict_proba(Xl_te)[:, 1])
print(f'"Leaky" AUC (adds second_to_first_ratio): {leaky_auc:.3f}  <- looks almost perfect, for the wrong reason')

# --- Remove the leak, keep the honest number ---
final_features = honest_features
print(f'\nFinal, honest feature set: {final_features}')
print(f'Final, honest AUC kept for reporting: {honest_auc:.3f}')


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named limitations of this slice

1. **Search-visibility proxy, not a business-outcome proxy.** `is_declining` is built purely from GSC impressions. A page can lose impressions while engagement/revenue per visit improves, or vice versa — this contract cannot see that without joining GA4-derived fields, which I deliberately excluded from this lane.
2. **Unbalanced panel.** `gsc_data_start` / `ga4_data_start` differ per client, so "2026-03" does not mean the same amount of history for every client — some clients may have little or no prior data in March, biasing the first-half aggregates for those clients toward zero/near-zero.
3. **Half-month noise.** Splitting a single month at the 15th halves an already-daily-noisy signal; the `imp_first_half >= 20` floor helps but doesn't eliminate small-sample volatility, especially for low-traffic pages.
4. **No forward validation possible from this slice alone.** The sealed `2026-06` sample month can't be touched to check whether March-trained logic generalizes forward — that check has to wait for a properly scoped, later evaluation, not something this contract can claim today.
5. **Client-level generalization untested here.** This notebook does not use a client-grouped train/test split (unlike the Week-2 lesson) — so the honest AUC above should be read as a same-month, mixed-client number, not a claim about performance on unseen clients.

In [ ]:
# Back one limitation with a number: how unbalanced is the panel, in terms of GSC history depth?
panel = con.sql(f"""
    SELECT gsc_data_start
    FROM {TABLES['dim_clients']}
    WHERE gsc_data_start IS NOT NULL
""").df()

panel['gsc_data_start'] = pd.to_datetime(panel['gsc_data_start'])
months_of_history = (pd.Timestamp('2026-03-31') - panel['gsc_data_start']).dt.days / 30

print(f'Clients with GSC history: {len(panel)}')
print(f'Shortest history as of 2026-03-31: {months_of_history.min():.1f} months')
print(f'Longest history as of 2026-03-31:  {months_of_history.max():.1f} months')
print(f'Median history as of 2026-03-31:   {months_of_history.median():.1f} months')
print('\nConfirms: this is an unbalanced panel — treat any cross-client comparison with care.')


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.